# Harborline Foods - Validation Notebook

**Purpose:** End-to-end proof harness for the Analytics Engineer III take-home assessment.
Runs the full pipeline - data generation, schema creation, Task 1 dimension fix, Task 2
conformed net sales - and shows before/after output inline.

**Execution engine:** DuckDB (in-process SQL; same ANSI syntax as PostgreSQL).
To run against Postgres.app locally, swap the connection cell with a psycopg2/SQLAlchemy URL.

**To reproduce:** run all cells top to bottom from a clean state.


## Setup - dependencies and paths

In [1]:
import sys, os, subprocess

for pkg in ["duckdb", "pandas"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--quiet"])

import duckdb
import pandas as pd

# Paths relative to notebook location
NB_DIR   = os.path.abspath(os.path.dirname("harborline_validation.ipynb"))
ROOT     = os.path.join(NB_DIR, "..")
DATA_DIR = os.path.abspath(os.path.join(ROOT, "data"))
SRC_DIR  = os.path.abspath(os.path.join(ROOT, "src"))

print(f"Root: {os.path.abspath(ROOT)}")
print(f"Data: {DATA_DIR}")


Root: /sessions/affectionate-peaceful-cori/mnt/job-search/blue-margin-assessment
Data: /sessions/affectionate-peaceful-cori/mnt/job-search/blue-margin-assessment/data


## Step 1 - Generate synthetic data

In [2]:
result = subprocess.run(
    [sys.executable, os.path.join(SRC_DIR, "generate_data.py")],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)


Generating Harborline synthetic data...
  wrote dim_locations.csv                     37 rows
  wrote toast_transactions.csv            14,316 rows
  wrote qu_transactions.csv                3,581 rows
  wrote r365_gl_entries.csv                6,734 rows
  wrote qu_location_crosswalk.csv              9 rows

Brand B locations (unmapped in dim_locations): 9
Done. dim_locations intentionally has NO qu_store_code -> Task 1 fixes it.



## Step 2 - Load into DuckDB

In [3]:
con = duckdb.connect()

con.execute(f"""
    CREATE TABLE dim_locations        AS SELECT * FROM read_csv_auto('{DATA_DIR}/dim_locations.csv', sample_size=-1);
    CREATE TABLE toast_transactions   AS SELECT * FROM read_csv_auto('{DATA_DIR}/toast_transactions.csv', sample_size=-1);
    CREATE TABLE qu_transactions      AS SELECT * FROM read_csv_auto('{DATA_DIR}/qu_transactions.csv', sample_size=-1);
    CREATE TABLE r365_gl_entries      AS SELECT * FROM read_csv_auto('{DATA_DIR}/r365_gl_entries.csv', sample_size=-1);
    CREATE TABLE qu_location_crosswalk AS SELECT * FROM read_csv_auto('{DATA_DIR}/qu_location_crosswalk.csv', sample_size=-1);
""")

for t in ["dim_locations","toast_transactions","qu_transactions","r365_gl_entries","qu_location_crosswalk"]:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<30} {n:>7,} rows")


  dim_locations                       37 rows
  toast_transactions              14,316 rows
  qu_transactions                  3,581 rows
  r365_gl_entries                  6,734 rows
  qu_location_crosswalk                9 rows


## Task 1 - Conformed dimension fix

### BEFORE: Brand B rows in dim_locations (no qu_store_code column)


In [4]:
before = con.execute("""
    SELECT location_key, location_name, brand, toast_loc_id
    FROM dim_locations WHERE brand = 'Brand B'
    ORDER BY location_key
""").df()
before


,location_key,location_name,brand,toast_loc_id
0,29,Cherry Creek Bistro,Brand B,None
1,30,LoDo Kitchen,Brand B,None
2,31,RiNo Tap,Brand B,None
3,32,Wash Park Cafe,Brand B,None
4,33,Berkeley Table,Brand B,None
5,34,Capitol Hill Counter,Brand B,None
6,35,Sloans Lake Grill,Brand B,None
7,36,Stapleton Eatery,Brand B,None
8,37,Belmar Social,Brand B,None


### Apply the fix - ALTER TABLE + UPDATE from crosswalk

In [5]:
con.execute("ALTER TABLE dim_locations ADD COLUMN qu_store_code VARCHAR(10);")
con.execute("""
    UPDATE dim_locations
    SET qu_store_code = x.qu_store_code
    FROM qu_location_crosswalk x
    WHERE dim_locations.location_key = x.location_key;
""")
print("Column added and backfilled.")


Column added and backfilled.


### AFTER: Brand B rows with qu_store_code populated

In [6]:
after = con.execute("""
    SELECT location_key, location_name, brand, toast_loc_id, qu_store_code
    FROM dim_locations WHERE brand = 'Brand B'
    ORDER BY location_key
""").df()
after


,location_key,location_name,brand,toast_loc_id,qu_store_code
0,29,Cherry Creek Bistro,Brand B,None,STR-B01
1,30,LoDo Kitchen,Brand B,None,STR-B02
2,31,RiNo Tap,Brand B,None,STR-B03
3,32,Wash Park Cafe,Brand B,None,STR-B04
4,33,Berkeley Table,Brand B,None,STR-B05
5,34,Capitol Hill Counter,Brand B,None,STR-B06
6,35,Sloans Lake Grill,Brand B,None,STR-B07
7,36,Stapleton Eatery,Brand B,None,STR-B08
8,37,Belmar Social,Brand B,None,STR-B09


### Validation (a) - Brand B mapping completeness

In [7]:
r = con.execute("""
    SELECT COUNT(*) AS brand_b_locations,
           COUNT(qu_store_code) AS mapped,
           COUNT(*) - COUNT(qu_store_code) AS still_missing
    FROM dim_locations WHERE brand = 'Brand B'
""").df()
display(r)
assert r['still_missing'][0] == 0, "FAIL: some Brand B locations still unmapped"
print("PASS: all 9 Brand B locations mapped, 0 still missing")


,brand_b_locations,mapped,still_missing
0,9,9,0


PASS: all 9 Brand B locations mapped, 0 still missing


### Validation (b) - No orphan Qu store codes

In [8]:
orphans = con.execute("""
    SELECT DISTINCT q.store_code
    FROM qu_transactions q
    LEFT JOIN dim_locations d ON d.qu_store_code = q.store_code
    WHERE d.qu_store_code IS NULL
""").df()
assert len(orphans) == 0, f"FAIL: {len(orphans)} orphan store codes"
n_codes = con.execute("SELECT COUNT(DISTINCT store_code) FROM qu_transactions").fetchone()[0]
print(f"PASS: 0 orphan Qu store codes (all {n_codes} codes resolve)")


PASS: 0 orphan Qu store codes (all 9 codes resolve)


## Task 2 - Conformed net sales

### Validation 1 - row count: conformed = Toast + Qu

In [9]:
counts = con.execute("""
    SELECT
      (SELECT COUNT(*) FROM toast_transactions) AS toast_rows,
      (SELECT COUNT(*) FROM qu_transactions)    AS qu_rows,
      (SELECT COUNT(*) FROM (
            SELECT 1 FROM toast_transactions t JOIN dim_locations d ON d.toast_loc_id = t.location_id
            UNION ALL
            SELECT 1 FROM qu_transactions q JOIN dim_locations d ON d.qu_store_code = q.store_code
       ) z) AS conformed_rows
""").df()
display(counts)
t = counts['toast_rows'][0]; q = counts['qu_rows'][0]; c = counts['conformed_rows'][0]
assert c == t + q, f"FAIL: {c} != {t} + {q}"
print(f"PASS: {c:,} = {t:,} (Toast) + {q:,} (Qu)")


,toast_rows,qu_rows,conformed_rows
0,14316,3581,17897


PASS: 17,897 = 14,316 (Toast) + 3,581 (Qu)


### Validation 2 - No orphan transactions

In [10]:
orphans = con.execute("""
    SELECT
      (SELECT COUNT(*) FROM toast_transactions t
         LEFT JOIN dim_locations d ON d.toast_loc_id = t.location_id
         WHERE d.location_key IS NULL) AS toast_orphans,
      (SELECT COUNT(*) FROM qu_transactions q
         LEFT JOIN dim_locations d ON d.qu_store_code = q.store_code
         WHERE d.location_key IS NULL) AS qu_orphans
""").df()
display(orphans)
assert orphans['toast_orphans'][0] == 0 and orphans['qu_orphans'][0] == 0
print("PASS: 0 orphan transactions in either source")


,toast_orphans,qu_orphans
0,0,0


PASS: 0 orphan transactions in either source


### Validation 3 - Net sales reconciliation

In [11]:
rec = con.execute("""
    SELECT
      ROUND(SUM(net_sales), 2) AS toast_net
    FROM toast_transactions
""").fetchone()[0]
qu = con.execute("""
    SELECT ROUND(SUM(gross_sales - promo_deductions), 2) AS qu_net
    FROM qu_transactions
""").fetchone()[0]
conf = con.execute("""
    SELECT ROUND(SUM(net_sales), 2) FROM (
      SELECT t.net_sales FROM toast_transactions t
      JOIN dim_locations d ON d.toast_loc_id = t.location_id
      UNION ALL
      SELECT q.gross_sales - q.promo_deductions
      FROM qu_transactions q
      JOIN dim_locations d ON d.qu_store_code = q.store_code
    ) z
""").fetchone()[0]

summary = pd.DataFrame({
    'source': ['Toast', 'Qu', 'Combined (independent)', 'Conformed model'],
    'net_sales': [rec, qu, round(float(rec)+float(qu),2), conf]
})
display(summary)

indep = round(float(rec) + float(qu), 2)
assert round(float(conf), 2) == indep, f"FAIL: ${conf:,.2f} != ${indep:,.2f}"
print(f"PASS: conformed ${float(conf):,.2f} ties to source totals")


,source,net_sales
0,Toast,5340200.47
1,Qu,1044574.54
2,Combined (independent),6384775.01
3,Conformed model,6384775.01


PASS: conformed $6,384,775.01 ties to source totals


### Validation 4 - Brand B presence (before / after) - the money shot

In [12]:
print("BEFORE fix - Brand B net_sales: $0.00  (9 locations orphaned, unreachable by join)")
print()

brand_summary = con.execute("""
    WITH cns AS (
        SELECT d.location_key, t.net_sales AS net_sales
        FROM toast_transactions t JOIN dim_locations d ON d.toast_loc_id = t.location_id
        UNION ALL
        SELECT d.location_key, q.gross_sales - q.promo_deductions
        FROM qu_transactions q JOIN dim_locations d ON d.qu_store_code = q.store_code
    )
    SELECT d.brand, COUNT(*) AS txns, ROUND(SUM(cns.net_sales), 2) AS net_sales
    FROM cns JOIN dim_locations d ON d.location_key = cns.location_key
    GROUP BY d.brand ORDER BY d.brand
""").df()

print("AFTER fix:")
display(brand_summary)

b_sales = float(brand_summary[brand_summary['brand']=='Brand B']['net_sales'].iloc[0])
assert b_sales > 0, "FAIL: Brand B still shows zero net sales after fix"
print(f"PASS: Brand B shows ${b_sales:,.2f} - fully represented in the conformed model")


BEFORE fix - Brand B net_sales: $0.00  (9 locations orphaned, unreachable by join)

AFTER fix:


,brand,txns,net_sales
0,Brand A,14316,5340200.47
1,Brand B,3581,1044574.54


PASS: Brand B shows $1,044,574.54 - fully represented in the conformed model


## Summary

| Check | Result |
|---|---|
| Brand B mapping completeness | 9/9 locations mapped, 0 still missing |
| Orphan Qu store codes | 0 |
| Conformed row count | 17,897 = 14,316 + 3,581 |
| Toast orphans | 0 |
| Qu orphans | 0 |
| Net sales reconciliation | $6,384,775.01 (conformed = Toast + Qu independent calc) |
| Brand B presence | $1,044,574.54 (was $0 before fix) |

All validations passed. The conformed model is ready for the Power BI layer.
